# Cohere Jupyter GUI - Quickstart

A Jupyter-native GUI for Bragg CDI reconstruction, alongside the PyQt5 `cohere_gui`.

## Install

```bash
pip install cohere-ui[jupyter]
```

The `[jupyter]` extra adds ipywidgets, ipympl, ipyfilechooser, and matplotlib. Headless / CLI / Qt-only users can install plain `pip install cohere-ui` to skip these.

## Use
1. Run the cells below in order.
2. Either browse to your experiment directory in the GUI's file picker, or call `gui.load_experiment('/path/to/experiment')` from a cell.
3. Configure each tab (Instrument, Prep, Data, Reconstruction, Display) and click the action button.
4. Live progress, snapshot images, and the error-history plot stream into the Reconstruction tab while a run is in progress.
5. Click **Stop** to kill a running reconstruction (process-group SIGKILL; works for single, multipeak, populous, multi-scan, and GA paths).

In [ ]:
from cohere_ui.jupyter_gui import CoherenceGUI

gui = CoherenceGUI()
gui.display()

## Programmatic loading (optional)

Set `EXP_DIR` below or in your shell environment to load an experiment without the file picker.

In [ ]:
gui.load_experiment('/path/to/your/experiment')
print('tabs:', list(gui._tabs.keys()))


## TIFF viewer & ImageJ integration

The **Prep Data** tab includes a side-by-side TIFF viewer that loads `preprocessed_data/prep_data.tif` (raw assembled stack) and `phasing_data/data.tif` (post-prep). Each pane has a slice slider; the **Sync scroll** checkbox keeps both aligned, **Log scale** applies `log10(I+1)` for diffraction-pattern detail. Pixels are rendered 1:1: no resampling, no interpolation, no upscale.

The toolbar also exposes:
- **Cmap** dropdown (magma / viridis / plasma / inferno / cividis / gray / hot / turbo / bone / cubehelix) and an **Invert cmap** checkbox
- **Intensity scale** radio: *Auto* (per slice), *Sync* (shared min/max across both panes, for comparable colors raw vs prep), *Manual* (per-pane vmin/vmax fields)

Each pane has an **ImageJ** button that opens the TIFF in ImageJ for full-fidelity inspection (orthoviews, ROI, profiles, etc.).

### How the ImageJ button finds your install

Resolution order:

1. **`cohere_ui.jupyter_gui.IMAGEJ_PATH`** (recommended escape hatch; see below)
2. **`IMAGEJ`** environment variable
3. **Standard install paths** by OS:
   - macOS: `/Applications`, `~/Applications`, `~/Downloads`, `~/Desktop`
   - Linux: `~/Applications/ImageJ-linux64`, `~/Downloads/...`, `/opt/imagej/...`, `/usr/local/...`
   - Windows: `C:\Program Files\ImageJ-win64.exe`, `%LOCALAPPDATA%\Programs\...`, `%USERPROFILE%\Downloads\...`
4. **macOS Spotlight (`mdfind`)** as a smart fallback that finds an ImageJ install wherever it lives (external volumes, Dropbox, etc.)
5. **`PATH` lookup** for `imagej`, `ImageJ-linux64`, `ImageJ-win64.exe`

If the button fails, the status text below the path field lists every path that was tried so you know exactly what to fix.

### Recommended: set `IMAGEJ_PATH` from this notebook

Run this **before** clicking the ImageJ button if auto-detection misses your install (or if you want to point at a specific install):

```python
import cohere_ui.jupyter_gui as cgui

# macOS: point at the .app bundle (uses `open -a`)
cgui.IMAGEJ_PATH = '/Users/me/Downloads/ImageJ.app'

# Linux: point at the binary
cgui.IMAGEJ_PATH = '/home/me/tools/ImageJ-linux64'

# Windows: point at the .exe
cgui.IMAGEJ_PATH = r'D:\tools\ImageJ-win64.exe'
```

`IMAGEJ_PATH` is checked first, ahead of every other resolution step.

### Alternative: shell environment variable

Set `IMAGEJ` before launching Jupyter:

```bash
# macOS / Linux
export IMAGEJ="/path/to/ImageJ.app"     # or .../ImageJ-linux64
jupyter lab

# Windows PowerShell
$env:IMAGEJ = 'D:\tools\ImageJ-win64.exe'
jupyter lab
```

Without ImageJ installed, the in-tab viewer still works fully; only the **ImageJ** buttons are inert.

## Inspecting results

After a reconstruction run completes, results are available on `gui.results`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gui.results.reload()
image = gui.results.image
support = gui.results.support
errors = gui.results.errors

if image is not None:
    mid = image.shape[0] // 2
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(np.abs(image[mid])); axes[0].set_title('Amplitude')
    axes[1].imshow(np.angle(image[mid]), cmap='twilight'); axes[1].set_title('Phase')
    if support is not None:
        axes[2].imshow(support[mid]); axes[2].set_title('Support')
    plt.tight_layout(); plt.show()
    print(f'Image shape: {image.shape}, dtype: {image.dtype}')
    if errors is not None:
        plt.figure(figsize=(8, 3))
        plt.semilogy(errors)
        plt.xlabel('iteration'); plt.ylabel('error (log)')
        plt.title(f'{len(errors)} iterations, final {errors[-1]:.4g}')
        plt.grid(True, alpha=0.3); plt.show()
else:
    print('No reconstruction results; run reconstruction first.')

## Interactive 3D inspection (PyVista)

After a reconstruction completes, the cell below loads `image`, `support`, and the diffraction `data` from `gui.results` and renders an interactive 3D view via PyVista's trame backend. You can rotate, pan, zoom, and adjust the isosurface threshold inside the notebook.

Requires the trame stack: `pip install cohere-ui[jupyter]` (already installed if you used the `[jupyter]` extra).

In [ ]:
import numpy as np
import pyvista as pv
from ipywidgets import VBox, HBox, FloatSlider, Dropdown, HTML

pv.set_jupyter_backend('trame')  # interactive trame backend

gui.results.reload()
image = gui.results.image
support = gui.results.support
data = gui.results.data
errors = gui.results.errors

if image is None:
    print('No reconstruction results yet; run a reconstruction first.')
else:
    amp = np.abs(image).astype(np.float32)
    phase = np.angle(image).astype(np.float32)
    peak = float(amp.max())

    # Build a UniformGrid carrying amplitude and phase as point arrays.
    grid = pv.ImageData()
    grid.dimensions = np.array(amp.shape) + 1
    grid.spacing = (1.0, 1.0, 1.0)
    grid.cell_data['amplitude'] = amp.flatten(order='F')
    grid.cell_data['phase'] = phase.flatten(order='F')
    if support is not None:
        grid.cell_data['support'] = support.astype(np.float32).flatten(order='F')
    grid = grid.cell_data_to_point_data()

    info = HTML(
        f'<b>image</b>: shape={image.shape}, dtype={image.dtype}, peak={peak:.4g}<br>'
        f'<b>support</b>: {None if support is None else support.shape}<br>'
        f'<b>errors</b>: {len(errors) if errors is not None else 0} iterations recorded<br>'
        f'<b>data</b> (raw diffraction): {None if data is None else data.shape}'
    )

    plotter = pv.Plotter(notebook=True, window_size=(720, 540))
    plotter.set_background('white')
    plotter.add_axes()

    iso_threshold = peak * 0.3
    iso = grid.contour([iso_threshold], scalars='amplitude')
    actor = plotter.add_mesh(
        iso, scalars='phase', cmap='twilight',
        clim=(-np.pi, np.pi), opacity=0.9,
        scalar_bar_args={'title': 'phase (rad)'},
    )

    display(info)
    plotter.show()

    # Also expose the full numpy arrays as kernel variables for further analysis:
    print('Variables now available in this kernel:')
    print('  image, support, data, errors, amp, phase, grid, plotter')


## Reciprocal-space inspection (PyVista)

Compares the **measured** diffraction (`gui.results.data`, what cohere fit) against the **reconstructed** reciprocal space $|\text{FFT}(\text{image})|^2$. If the reconstruction converged, the two isosurfaces should overlap. Both are rendered as 3D isosurfaces of `log10(I+1)` so the dynamic range collapses into something the eye can read.

Linked subplots: rotating one rotates the other.

Variables exposed: `data_recip`, `recip_recon`, `residual` (all `numpy.ndarray`).

In [ ]:
import numpy as np
import pyvista as pv

pv.set_jupyter_backend('trame')

data_recip = gui.results.data
if data_recip is None or image is None:
    print('Need both gui.results.data and gui.results.image; run a reconstruction first.')
else:
    # Reconstructed reciprocal space and residual.
    recip_recon = np.fft.fftshift(np.abs(np.fft.fftn(image)) ** 2)
    if recip_recon.shape != data_recip.shape:
        recip_recon = np.transpose(recip_recon, (2, 1, 0))
    residual = np.abs(data_recip - recip_recon).astype(np.float32)

    def _grid_log(volume, name='intensity'):
        v = np.log10(np.maximum(np.asarray(volume, dtype=np.float32), 0) + 1.0)
        g = pv.ImageData()
        g.dimensions = np.array(v.shape) + 1
        g.spacing = (1.0, 1.0, 1.0)
        g.cell_data[name] = v.flatten(order='F')
        return g.cell_data_to_point_data(), float(v.max())

    def _add_layered(plotter, grid, peak, cmap):
        # Multi-level nested isosurfaces: outer translucent, inner opaque.
        # Best for sharp, structured peaks (measured Bragg with fringes).
        levels = [0.20, 0.35, 0.50, 0.70, 0.85]
        opacities = [0.10, 0.18, 0.30, 0.55, 0.95]
        for lvl, op in zip(levels, opacities):
            iso = grid.contour([peak * lvl], scalars='intensity')
            if iso.n_points == 0:
                continue
            plotter.add_mesh(
                iso, scalars='intensity', cmap=cmap,
                clim=(peak * 0.15, peak),
                opacity=op, show_scalar_bar=False,
            )

    def _add_single(plotter, grid, peak, color, level=0.55):
        # Single high-threshold contour: best for diffuse structures
        # (the reconstruction's reciprocal space) where layered isosurfaces
        # would just produce visual noise.
        iso = grid.contour([peak * level], scalars='intensity')
        if iso.n_points > 0:
            plotter.add_mesh(iso, color=color, opacity=0.7,
                             show_scalar_bar=False)

    g_meas, peak_meas = _grid_log(data_recip)
    g_rec, peak_rec = _grid_log(recip_recon)
    g_res, peak_res = _grid_log(residual)

    plotter = pv.Plotter(notebook=True, shape=(1, 3), window_size=(1280, 480))

    plotter.subplot(0, 0)
    plotter.set_background('white')
    _add_layered(plotter, g_meas, peak_meas, cmap='inferno')
    plotter.add_text('Measured (log10 I)', font_size=10, color='black')
    plotter.add_axes()

    plotter.subplot(0, 1)
    plotter.set_background('white')
    _add_single(plotter, g_rec, peak_rec, color='crimson', level=0.55)
    plotter.add_text('|FFT(image)|^2 (log10)', font_size=10, color='black')
    plotter.add_axes()

    plotter.subplot(0, 2)
    plotter.set_background('white')
    _add_single(plotter, g_res, peak_res, color='darkorange', level=0.4)
    plotter.add_text('|measured - recon|', font_size=10, color='black')
    plotter.add_axes()

    plotter.link_views()
    plotter.show()

    print('Reciprocal-space variables exposed in this kernel:')
    print(f'  data_recip   {data_recip.shape}  (measured intensity)')
    print(f'  recip_recon  {recip_recon.shape}  (|FFT(image)|^2 of converged reconstruction)')
    print(f'  residual     {residual.shape}  (|measured - reconstructed|)')
    print(f'  peaks (log10): measured={peak_meas:.3f}, recon={peak_rec:.3f}, residual={peak_res:.3f}')


## Direct tab/feature access

All tabs and features are accessible programmatically:

In [ ]:
print('rec config:', gui._tabs['rec'].get_config())
print('data config:', gui._tabs['data'].get_config())